# S0002 positive-only FFT review
Read-only audit of 365 supplied speech recordings. Energy-selected frames are not speech labels. Rules tested below are illustrative rejection criteria, not proposed production thresholds.


In [1]:
from pathlib import Path
import csv, json, math, hashlib
base=Path('D:/Espressif/projects/VOICE DATA BENCHMARK/analysis/S0002_fft')
summary_path=base/'file_summary.csv'
rows=list(csv.DictReader(summary_path.open(encoding='utf-8-sig')))
active=[];full=0
for p in sorted((base/'per_file').glob('*.frames.csv')):
    for r in csv.DictReader(p.open(encoding='utf-8-sig')):
        if r['complete']!='True':continue
        full+=1
        if r['active']=='True':active.append(r)
valid=[r for r in active if math.isfinite(float(r['f0_hz']))]
def pct(n,d):return round(100*n/d,2)
result={'files':len(rows),'duration_s':sum(float(r['duration_s']) for r in rows),
 'full_frames':full,'active_frames':len(active),'active_threshold_dbfs':-50,
 'active_with_f0':len(valid),
 'active_without_f0_pct':pct(len(active)-len(valid),len(active)),
 'active_centroid_above1500_pct':pct(sum(float(r['centroid_hz'])>1500 for r in active),len(active)),
 'active_rolloff_above1500_pct':pct(sum(float(r['rolloff95_hz'])>1500 for r in active),len(active)),
 'active_peak_outside150_250_pct':pct(sum(not 150<=float(r['peak_hz'])<=250 for r in active),len(active)),
 'file_0_1000_pct_range':[min(float(r['band_0_300_pct'])+float(r['band_300_1000_pct']) for r in rows),max(float(r['band_0_300_pct'])+float(r['band_300_1000_pct']) for r in rows)],
 'files_rolloff_above1500':sum(float(r['rolloff95_hz'])>1500 for r in rows),
 'file_summary_sha256':hashlib.sha256(summary_path.read_bytes()).hexdigest(),
 'scope':'Speech-recording frames, not frame-level ground-truth speech labels. No negative noise recordings. Rejection percentages are NOT speech false-rejection rates.'}
assert len(rows)==365 and full==87865 and len(active)==51557
print(json.dumps(result,indent=2))


{
  "files": 365,
  "duration_s": 1760.9918125,
  "full_frames": 87865,
  "active_frames": 51557,
  "active_threshold_dbfs": -50,
  "active_with_f0": 25100,
  "active_without_f0_pct": 51.32,
  "active_centroid_above1500_pct": 10.12,
  "active_rolloff_above1500_pct": 23.25,
  "active_peak_outside150_250_pct": 51.21,
  "file_0_1000_pct_range": [
    79.10953457669947,
    98.31403098179113
  ],
  "files_rolloff_above1500": 163,
  "file_summary_sha256": "9a6e0b93756ee18e70fce00d6bd5a57b4f2f4043e1efd94c360e1cff94b6ad17",
  "scope": "Speech-recording frames, not frame-level ground-truth speech labels. No negative noise recordings. Rejection percentages are NOT speech false-rejection rates."
}
